Script for lock in:

 parameter: r (resistance)

 - Set V (Read) (can calculate I=V/r)
 - Set frequency (Read)
 - Read R ($\sqrt(X^2+Y^2)$), $\theta$ (r = R / I)
 - Voltage: A_B (Read)
 - Other: time constant, input range, sensitivity

 Read lock in (R, $\theta$) ever x seconds 
 Save to text file (also need to read temperatures from MXC) as MXC Temperature, Resistance (Ohms), Theta

Read MXC:
LakeShore

In [ ]:
import pyvisa
from pyvisa import VisaIOError
from datetime import datetime
import time

In [3]:
rm = pyvisa.ResourceManager()

for s in rm.list_resources():
    try:
        lockin = rm.open_resource(s)
        print(s, lockin.query('*IDN?').strip())
    except VisaIOError:
        continue

GPIB0::4::INSTR Stanford_Research_Systems,SR860,005576,V1.51


In [24]:
rm = pyvisa.ResourceManager()
lockin = rm.open_resource('GPIB0::4::INSTR')  # where the Lockin is

print("Connected to:", lockin.query("*IDN?").strip())

Rbias = 1e6

Connected to: Stanford_Research_Systems,SR860,005576,V1.51


In [6]:
def set_voltage(voltage): # ex set_voltage(0.100) sets to 100 mV
    lockin.write(f"SLVL {voltage}")

def read_voltage(): # reads frequency
    return float(lockin.query("SLVL?"))

In [ ]:
def set_frequency(freq):
    lockin.write(f"FREQ {freq}")

def read_frequency():
    return float(lockin.query("FREQ?"))

In [8]:
def read_x():# reads x
    return float(lockin.query("OUTP? 0"))

In [9]:
def read_y(): #reads y
    return float(lockin.query("OUTP? 1"))

In [10]:
def read_r():
    return float(lockin.query("OUTP? 2"))

In [11]:
def read_theta():
    return float(lockin.query("OUTP? 3"))

In [35]:
def set_output(num):
    lockin.write(f"ISRC {num}")

In [13]:
def calculate_current():
    V = read_voltage()
    return V / Rbias

In [14]:
def calculate_sample_resistance():
    voltage = read_r()
    current = calculate_current()
    return voltage / current # == R = I/V

In [ ]:
def log_data(filename):
    with open(filename, "a") as file:
        i = 0
        while i < 5:
            # timestamp = datetime.now()
            # freq = read_frequency()
            # V = read_voltage()
            # current = V / resistor
            # R = read_r()
            theta = read_theta()
            sampleR = calculate_sample_resistance()
            # AB = read_ab()

            file.write(
                # f"{timestamp,}"
                # f"{freq},"
                # f"{V},"
                # f"{current},"
                # f"{R},"
                f"{theta},"
                f"{sampleR},"
                # f"{AB}\n"
            )
            file.flush()
            # print(timestamp, R, theta)
            time.sleep(1)
            i+=1

In [45]:
log_data("C:\\Users\\MINER\\Documents\\B13 Cryolab\\Test.txt")